# A2 — Linear vs cosine noise schedule cho DDPM

**Đề tài:** ảnh hưởng của noise schedule tới mô hình sinh khuôn mặt không điều kiện ở 32×32.

Paper chính: *Improved Denoising Diffusion Probabilistic Models* — Nichol & Dhariwal, ICML 2021. Thực nghiệm này chỉ tái hiện ablation linear/cosine schedule; không tái hiện learned variance hoặc toàn bộ kết quả paper.


## 1. Môi trường


In [ ]:
import importlib.util, subprocess, sys

required = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready")

extra = {"diffusers": "diffusers>=0.30", "accelerate": "accelerate>=0.33",
         "torchmetrics": "torchmetrics[image]>=1.4", "torch_fidelity": "torch-fidelity>=0.3"}
missing_extra = [package for module, package in extra.items() if importlib.util.find_spec(module) is None]
if missing_extra:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_extra])


In [ ]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
print("device:", DEVICE)

from PIL import Image
from diffusers import DDPMScheduler, UNet2DModel
from torch.utils.data import Sampler
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchvision.utils import make_grid


## 2. Cấu hình


In [ ]:
def default_root(kind):
    env_name = "CO5085_DATA_ROOT" if kind == "data" else "CO5085_OUTPUT_ROOT"
    if os.getenv(env_name):
        return Path(os.environ[env_name])
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / ("data" if kind == "data" else "artifacts")
    if Path("/content").exists():
        return Path("/content") / ("data" if kind == "data" else "artifacts")
    return Path("./data" if kind == "data" else "./artifacts")


CONFIG = {
    "data_root": str(default_root("data")),
    "output_root": str(default_root("output")),
    "run_mode": os.getenv("CO5085_RUN_MODE", "smoke"),
    "experiment_ids": [],  # Empty means all experiments for this notebook.
    "seed": 42,
    "batch_size": 128,
    "epochs": 20,
    "resume_from": None,  # None, "last", or a checkpoint path.
    "evaluate_only": os.getenv("CO5085_EVALUATE_ONLY", "0") == "1",
    "num_workers": 2,
}
assert CONFIG["run_mode"] in {"smoke", "full"}
if CONFIG["run_mode"] == "smoke":
    CONFIG["epochs"] = 1
    CONFIG["num_workers"] = 0
print(json.dumps(CONFIG, indent=2))

CONFIG.update({
    "celeba_root": os.getenv("CO5085_CELEBA_ROOT", str(Path(CONFIG["data_root"]) / "celeba")),
    "batch_size": 32, "effective_batch_size": 64, "gradient_accumulation": 2,
    "max_steps": 2 if CONFIG["run_mode"] == "smoke" else 20000,
    "save_every": 1 if CONFIG["run_mode"] == "smoke" else 1000,
    "preview_inference_steps": 10 if CONFIG["run_mode"] == "smoke" else 50,
    "eval_inference_steps": 10 if CONFIG["run_mode"] == "smoke" else 250,
    "eval_samples": 64 if CONFIG["run_mode"] == "smoke" else 5000,
})
CONFIG["num_workers"] = 0  # Required for exact sampler resume.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def environment_info():
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "sklearn": sklearn.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }


def json_dump(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, default=str)


def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**20
    return {"parameters": total, "trainable_parameters": trainable, "weight_size_mb": size_mb}


def split_digest(train_idx, val_idx):
    digest = hashlib.sha256()
    digest.update(np.asarray(train_idx, dtype=np.int64).tobytes())
    digest.update(np.asarray(val_idx, dtype=np.int64).tobytes())
    return digest.hexdigest()


def fashion_split(dataset, data_root, seed=42):
    split_path = Path(data_root) / "fashion_mnist_split_seed42.npz"
    labels = np.asarray(dataset.targets)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=6000, random_state=seed)
    expected_train, expected_val = next(splitter.split(np.zeros(len(labels)), labels))
    if split_path.exists():
        saved = np.load(split_path)
        train_idx, val_idx = saved["train_idx"], saved["val_idx"]
        if not (np.array_equal(train_idx, expected_train) and np.array_equal(val_idx, expected_val)):
            raise RuntimeError(f"Split file does not match seed={seed}: {split_path}")
    else:
        split_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(split_path, train_idx=expected_train, val_idx=expected_val)
        train_idx, val_idx = expected_train, expected_val
    assert len(train_idx) == 54000 and len(val_idx) == 6000
    assert not set(train_idx).intersection(set(val_idx))
    return train_idx, val_idx, split_digest(train_idx, val_idx)


def classification_epoch(model, loader, criterion, optimizer=None, scaler=None,
                         max_batches=None, grad_clip=None, grad_accum=1, frozen_eval=None):
    training = optimizer is not None
    model.train(training)
    if training and frozen_eval is not None:
        frozen_eval(model)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = total_correct = total_items = 0
    accumulation_count = 0
    start = time.perf_counter()
    context = torch.enable_grad if training else torch.no_grad
    with context():
        for batch_index, (images, labels) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                logits = model(images)
                loss = criterion(logits, labels)
                backward_loss = loss / grad_accum
            if training:
                scaler.scale(backward_loss).backward()
                accumulation_count += 1
                should_step = accumulation_count == grad_accum
                if should_step:
                    if grad_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    accumulation_count = 0
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_items += labels.size(0)
        if training and accumulation_count:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    elapsed = time.perf_counter() - start
    return {"loss": total_loss / total_items, "accuracy": total_correct / total_items,
            "items": total_items, "seconds": elapsed}


@torch.no_grad()
def predict_classifier(model, loader, max_batches=None):
    model.eval()
    labels_all, predictions_all, probabilities_all = [], [], []
    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        logits = model(images.to(DEVICE))
        probabilities = logits.softmax(1).cpu()
        labels_all.append(labels.cpu())
        predictions_all.append(probabilities.argmax(1))
        probabilities_all.append(probabilities)
    return (torch.cat(labels_all).numpy(), torch.cat(predictions_all).numpy(),
            torch.cat(probabilities_all).numpy())


def resolve_resume(run_dir, resume_from):
    if resume_from is None:
        return None
    if resume_from == "last":
        candidate = Path(run_dir) / "last.pt"
    else:
        candidate = Path(resume_from)
    return candidate if candidate.exists() else None


def fit_classifier(model, loaders, optimizer, scheduler, run_dir, run_config,
                   grad_clip=None, grad_accum=1, frozen_eval=None):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    json_dump(run_config, run_dir / "config.json")
    json_dump(environment_info(), run_dir / "environment.json")
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    start_epoch, best_accuracy, best_loss, history = 0, -1.0, float("inf"), []
    if run_config.get("evaluate_only"):
        evaluation_path = run_dir / "best.pt"
        if not evaluation_path.exists():
            raise FileNotFoundError(f"evaluate_only requires {evaluation_path}")
        checkpoint = torch.load(evaluation_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model"])
        start_epoch = run_config["epochs"]
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")
    else:
        resume_path = resolve_resume(run_dir, run_config.get("resume_from"))
        checkpoint = torch.load(resume_path, map_location=DEVICE) if resume_path is not None else None
    if not run_config.get("evaluate_only") and checkpoint is not None:
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if scheduler is not None and checkpoint.get("scheduler") is not None:
            scheduler.load_state_dict(checkpoint["scheduler"])
        if checkpoint.get("scaler") is not None:
            scaler.load_state_dict(checkpoint["scaler"])
        restore_rng_state(checkpoint["rng_state"])
        start_epoch = checkpoint["epoch"] + 1
        best_accuracy, best_loss = checkpoint["best_accuracy"], checkpoint["best_loss"]
        history_path = run_dir / "history.csv"
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(start_epoch, run_config["epochs"]):
        train_stats = classification_epoch(
            model, loaders["train"], criterion, optimizer, scaler,
            max_batches=run_config.get("max_train_batches"), grad_clip=grad_clip,
            grad_accum=grad_accum, frozen_eval=frozen_eval,
        )
        val_stats = classification_epoch(
            model, loaders["val"], criterion,
            max_batches=run_config.get("max_eval_batches"),
        )
        if scheduler is not None:
            scheduler.step()
        row = {
            "epoch": epoch,
            "train_loss": train_stats["loss"], "train_accuracy": train_stats["accuracy"],
            "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
            "epoch_seconds": train_stats["seconds"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        improved = (val_stats["accuracy"] > best_accuracy or
                    (val_stats["accuracy"] == best_accuracy and val_stats["loss"] < best_loss))
        if improved:
            best_accuracy, best_loss = val_stats["accuracy"], val_stats["loss"]
        state = {
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "scaler": scaler.state_dict(), "epoch": epoch,
            "best_accuracy": best_accuracy, "best_loss": best_loss,
            "rng_state": rng_state(), "config": run_config,
        }
        atomic_torch_save(state, run_dir / "last.pt")
        if improved:
            atomic_torch_save(state, run_dir / "best.pt")
        print(f"epoch={epoch + 1}/{run_config['epochs']} train={train_stats['accuracy']:.4f} val={val_stats['accuracy']:.4f}")

    best = torch.load(run_dir / "best.pt", map_location=DEVICE)
    model.load_state_dict(best["model"])
    val_stats = classification_epoch(model, loaders["val"], criterion,
                                     max_batches=run_config.get("max_eval_batches"))
    test_stats = classification_epoch(model, loaders["test"], criterion,
                                      max_batches=run_config.get("max_eval_batches"))
    metrics = {
        "run_mode": run_config["run_mode"], "best_epoch": best["epoch"],
        "val_loss": val_stats["loss"], "val_accuracy": val_stats["accuracy"],
        "test_loss": test_stats["loss"], "test_accuracy": test_stats["accuracy"],
        "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                               if torch.cuda.is_available() else 0.0),
        **count_parameters(model),
    }
    y_true, y_pred, probabilities = predict_classifier(
        model, loaders["test"], max_batches=run_config.get("max_eval_batches"))
    np.savez_compressed(run_dir / "predictions.npz", y_true=y_true, y_pred=y_pred,
                        probabilities=probabilities)
    json_dump(metrics, run_dir / "metrics.json")
    return metrics, pd.DataFrame(history), (y_true, y_pred, probabilities)


@torch.no_grad()
def benchmark_model(model, sample, repeats=30, warmup=5):
    model.eval()
    sample = sample.to(DEVICE)
    for _ in range(warmup):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    for _ in range(repeats):
        model(sample)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "latency_ms_per_image": 1000 * elapsed / (repeats * len(sample)),
        "benchmark_batch_size": len(sample),
        "benchmark_peak_memory_mb": (torch.cuda.max_memory_allocated() / 2**20
                                     if torch.cuda.is_available() else 0.0),
    }


def plot_history(history, path):
    frame = pd.DataFrame(history)
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(frame.epoch, frame.train_loss, label="train")
    axes[0].plot(frame.epoch, frame.val_loss, label="validation")
    axes[0].set(title="Loss", xlabel="Epoch")
    axes[1].plot(frame.epoch, frame.train_accuracy, label="train")
    axes[1].plot(frame.epoch, frame.val_accuracy, label="validation")
    axes[1].set(title="Accuracy", xlabel="Epoch")
    for axis in axes:
        axis.legend(); axis.grid(alpha=.2)
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def plot_confusion(y_true, y_pred, class_names, path, normalize=True):
    matrix = confusion_matrix(y_true, y_pred, normalize="true" if normalize else None)
    size = max(7, min(20, len(class_names) * .22))
    figure, axis = plt.subplots(figsize=(size, size))
    sns.heatmap(matrix, cmap="Blues", xticklabels=class_names, yticklabels=class_names,
                annot=len(class_names) <= 10, fmt=".2f" if normalize else "d", ax=axis)
    axis.set(xlabel="Predicted", ylabel="True", title="Confusion matrix")
    figure.tight_layout(); figure.savefig(path, dpi=160); plt.close(figure)


def completed_metrics(run_dir, run_mode):
    path = Path(run_dir) / "metrics.json"
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as handle:
        metrics = json.load(handle)
    return metrics if metrics.get("run_mode") == run_mode else None


seed_everything(CONFIG["seed"])
Path(CONFIG["data_root"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)


## 3. Dữ liệu CelebA và điều khoản sử dụng

CelebA chỉ được dùng cho nghiên cứu phi thương mại. Không commit hoặc phân phối lại ảnh. Mount bản Align&Cropped cùng `list_eval_partition.txt`, sau đó đặt `CO5085_CELEBA_ROOT`. Notebook lưu manifest filename để tái tạo subset.


In [ ]:
CELEBA_ROOT = Path(CONFIG["celeba_root"])
IMAGE_ROOT = CELEBA_ROOT / "img_align_celeba"
PARTITION_FILE = CELEBA_ROOT / "list_eval_partition.txt"
if not IMAGE_ROOT.exists() or not PARTITION_FILE.exists():
    raise FileNotFoundError(
        "Mount CelebA Align&Cropped, including list_eval_partition.txt, then set "
        "CO5085_CELEBA_ROOT to that directory. Dataset images must not be committed to the repo."
    )

partition_frame = pd.read_csv(PARTITION_FILE, sep=r"\s+", names=["filename", "partition"])
requested = {"train": (0, 20000), "val": (1, 2000), "test": (2, 5000)}
manifest_rows = []
for offset, (split_name, (partition_id, count)) in enumerate(requested.items()):
    candidates = partition_frame.loc[partition_frame.partition == partition_id, "filename"].to_numpy()
    rng = np.random.default_rng(CONFIG["seed"] + offset)
    chosen = candidates[rng.permutation(len(candidates))[:count]]
    manifest_rows.extend({"filename": name, "split": split_name} for name in chosen)
manifest = pd.DataFrame(manifest_rows)
assert len(manifest) == 27000 and manifest.filename.is_unique
MANIFEST_HASH = hashlib.sha256("\n".join(manifest.filename).encode()).hexdigest()
manifest_path = Path(CONFIG["output_root"]) / "A2" / "celeba_manifest_seed42.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
if manifest_path.exists():
    saved_manifest = pd.read_csv(manifest_path)
    if not saved_manifest.equals(manifest):
        raise RuntimeError("Existing CelebA manifest does not match deterministic selection")
else:
    manifest.to_csv(manifest_path, index=False)

train_transform = transforms.Compose([
    transforms.CenterCrop(148), transforms.Resize((32, 32), antialias=True),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize((.5, .5, .5), (.5, .5, .5)),
])
eval_transform = transforms.Compose([
    transforms.CenterCrop(148), transforms.Resize((32, 32), antialias=True),
    transforms.ToTensor(), transforms.Normalize((.5, .5, .5), (.5, .5, .5)),
])


class CelebASubset(Dataset):
    def __init__(self, frame, image_root, transform):
        self.filenames = frame.filename.tolist(); self.image_root = Path(image_root); self.transform = transform
    def __len__(self): return len(self.filenames)
    def __getitem__(self, index):
        with Image.open(self.image_root / self.filenames[index]) as image:
            return self.transform(image.convert("RGB"))


split_frames = {name: manifest[manifest.split == name].reset_index(drop=True) for name in requested}
train_dataset = CelebASubset(split_frames["train"], IMAGE_ROOT, train_transform)
val_dataset = CelebASubset(split_frames["val"], IMAGE_ROOT, eval_transform)
test_dataset = CelebASubset(split_frames["test"], IMAGE_ROOT, eval_transform)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
print({"sizes": [len(train_dataset), len(val_dataset), len(test_dataset)], "manifest_hash": MANIFEST_HASH})


## 4. Stateful sampler, EMA, mô hình U-Net và noise scheduler


In [ ]:
class StatefulRandomSampler(Sampler):
    def __init__(self, data_source, seed):
        self.data_source = data_source
        self.generator = torch.Generator().manual_seed(seed)
        self.permutation = None
        self.position = 0
    def __len__(self): return len(self.data_source)
    def __iter__(self):
        if self.permutation is None or self.position >= len(self.permutation):
            self.permutation = torch.randperm(len(self.data_source), generator=self.generator).tolist()
            self.position = 0
        while self.position < len(self.permutation):
            index = self.permutation[self.position]
            self.position += 1
            yield index
    def state_dict(self):
        return {"permutation": self.permutation, "position": self.position,
                "generator_state": self.generator.get_state()}
    def load_state_dict(self, state):
        self.permutation = state["permutation"]
        self.position = state["position"]
        self.generator.set_state(state["generator_state"])


class EMA:
    def __init__(self, model, decay=.999):
        self.decay = decay
        self.shadow = {name: parameter.detach().clone() for name, parameter in model.named_parameters()}
        self.backup = None
    @torch.no_grad()
    def update(self, model):
        for name, parameter in model.named_parameters():
            self.shadow[name].mul_(self.decay).add_(parameter.detach(), alpha=1 - self.decay)
    def state_dict(self): return {"decay": self.decay, "shadow": self.shadow}
    def load_state_dict(self, state): self.decay, self.shadow = state["decay"], state["shadow"]
    @torch.no_grad()
    def apply(self, model):
        self.backup = {name: parameter.detach().clone() for name, parameter in model.named_parameters()}
        for name, parameter in model.named_parameters(): parameter.copy_(self.shadow[name])
    @torch.no_grad()
    def restore(self, model):
        for name, parameter in model.named_parameters(): parameter.copy_(self.backup[name])
        self.backup = None


def build_unet():
    seed_everything(CONFIG["seed"])
    return UNet2DModel(
        sample_size=32, in_channels=3, out_channels=3, layers_per_block=2,
        block_out_channels=(64, 128, 128),
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
        up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
        norm_num_groups=32,
    )


def build_noise_scheduler(schedule):
    arguments = dict(num_train_timesteps=1000, variance_type="fixed_small",
                     prediction_type="epsilon", clip_sample=True)
    if schedule == "linear":
        return DDPMScheduler(beta_schedule="linear", beta_start=1e-4, beta_end=.02, **arguments)
    return DDPMScheduler(beta_schedule="squaredcos_cap_v2", **arguments)


EXPERIMENT_IDS = ["linear", "cosine"]
selected = CONFIG["experiment_ids"] or EXPERIMENT_IDS
unknown = set(selected) - set(EXPERIMENT_IDS)
if unknown: raise ValueError(f"Unknown experiment IDs: {sorted(unknown)}")

reference_states = []
for schedule_name in EXPERIMENT_IDS:
    scheduler = build_noise_scheduler(schedule_name)
    assert torch.isfinite(scheduler.betas).all() and (scheduler.betas > 0).all() and (scheduler.betas < 1).all()
    model = build_unet()
    reference_states.append({name: value.clone() for name, value in model.state_dict().items()})
    clean, noise, timesteps = torch.randn(2, 3, 32, 32), torch.randn(2, 3, 32, 32), torch.tensor([0, 999])
    noisy = scheduler.add_noise(clean, noise, timesteps)
    prediction = model(noisy, timesteps).sample
    reverse = scheduler.step(prediction, 999, noisy).prev_sample
    assert prediction.shape == clean.shape and torch.isfinite(noisy).all() and torch.isfinite(reverse).all()
for name in reference_states[0]:
    torch.testing.assert_close(reference_states[0][name], reference_states[1][name], rtol=0, atol=0)
print("U-Net initialization, add-noise and reverse-step checks passed")
del model, reference_states


## 5. Vòng lặp huấn luyện và checkpoint theo optimizer step


In [ ]:
@torch.no_grad()
def validation_noise_loss(model, scheduler, loader, max_batches=20):
    model.eval(); losses = []
    generator = torch.Generator(device=DEVICE.type).manual_seed(CONFIG["seed"] + 999)
    for batch_index, clean in enumerate(loader):
        if batch_index >= max_batches: break
        clean = clean.to(DEVICE)
        noise = torch.randn(clean.shape, generator=generator, device=DEVICE)
        timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (len(clean),),
                                  generator=generator, device=DEVICE)
        noisy = scheduler.add_noise(clean, noise, timesteps)
        losses.append(F.mse_loss(model(noisy, timesteps).sample, noise).item())
    return float(np.mean(losses))


@torch.no_grad()
def sample_images(model, scheduler, initial_noise, inference_steps, return_trace=False):
    sampler = DDPMScheduler.from_config(scheduler.config)
    sampler.set_timesteps(inference_steps, device=DEVICE)
    images, trace = initial_noise.to(DEVICE), []
    interval = max(1, len(sampler.timesteps) // 8)
    for index, timestep in enumerate(sampler.timesteps):
        predicted_noise = model(images, timestep).sample
        images = sampler.step(predicted_noise, timestep, images).prev_sample
        if return_trace and (index % interval == 0 or index == len(sampler.timesteps) - 1):
            trace.append(images.detach().cpu())
    return images.clamp(-1, 1), trace


def save_preview(model, ema, scheduler, run_dir, step):
    ema.apply(model); model.eval()
    generator = torch.Generator().manual_seed(CONFIG["seed"] + 123)
    initial = torch.randn(16, 3, 32, 32, generator=generator)
    images, _ = sample_images(model, scheduler, initial, CONFIG["preview_inference_steps"])
    ema.restore(model)
    grid = make_grid((images.cpu() + 1) / 2, nrow=4)
    figure, axis = plt.subplots(figsize=(6, 6)); axis.imshow(grid.permute(1, 2, 0)); axis.axis("off")
    figure.tight_layout(); figure.savefig(Path(run_dir) / f"preview_step_{step:06d}.png", dpi=160); plt.close(figure)


def train_diffusion(experiment_id):
    run_dir = Path(CONFIG["output_root"]) / "A2" / CONFIG["run_mode"] / experiment_id
    run_dir.mkdir(parents=True, exist_ok=True)
    run_config = {**CONFIG, "assignment": "A2", "experiment_id": experiment_id,
                  "noise_schedule": experiment_id, "manifest_hash": MANIFEST_HASH,
                  "train_samples": len(train_dataset), "timesteps": 1000,
                  "prediction_target": "epsilon", "variance_type": "fixed_small"}
    json_dump(run_config, run_dir / "config.json"); json_dump(environment_info(), run_dir / "environment.json")
    model = build_unet().to(DEVICE)
    scheduler = build_noise_scheduler(experiment_id)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    lr_scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min(1.0, (step + 1) / 500) if step < 500 else
        .5 * (1 + math.cos(math.pi * (step - 500) / max(1, CONFIG["max_steps"] - 500))))
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    ema = EMA(model, .999)
    sampler = StatefulRandomSampler(train_dataset, CONFIG["seed"])
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], sampler=sampler,
                              num_workers=0, drop_last=True, pin_memory=AMP_ENABLED)
    global_step, history = 0, []
    if CONFIG["evaluate_only"]:
        resume_path = run_dir / "last.pt"
        if not resume_path.exists():
            raise FileNotFoundError(f"evaluate_only requires {resume_path}")
    else:
        resume_path = resolve_resume(run_dir, CONFIG["resume_from"])
    if resume_path is not None:
        checkpoint = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model"]); optimizer.load_state_dict(checkpoint["optimizer"])
        lr_scheduler.load_state_dict(checkpoint["scheduler"]); scaler.load_state_dict(checkpoint["scaler"])
        ema.load_state_dict(checkpoint["ema"]); sampler.load_state_dict(checkpoint["sampler"])
        restore_rng_state(checkpoint["rng_state"]); global_step = checkpoint["global_step"]
        history_path = run_dir / "history.csv"
        if history_path.exists(): history = pd.read_csv(history_path).to_dict("records")
    if CONFIG["evaluate_only"]:
        return model, ema, scheduler, run_dir, history
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    optimizer.zero_grad(set_to_none=True); accumulation = 0
    progress = tqdm(total=CONFIG["max_steps"], initial=global_step, desc=experiment_id)
    while global_step < CONFIG["max_steps"]:
        for clean in train_loader:
            clean = clean.to(DEVICE)
            noise = torch.randn_like(clean)
            timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (len(clean),), device=DEVICE)
            noisy = scheduler.add_noise(clean, noise, timesteps)
            with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
                prediction = model(noisy, timesteps).sample
                loss = F.mse_loss(prediction, noise) / CONFIG["gradient_accumulation"]
            scaler.scale(loss).backward(); accumulation += 1
            if accumulation < CONFIG["gradient_accumulation"]: continue
            scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            lr_scheduler.step(); ema.update(model); accumulation = 0; global_step += 1
            progress.update(1)
            history.append({"global_step": global_step,
                            "train_loss": loss.item() * CONFIG["gradient_accumulation"],
                            "learning_rate": optimizer.param_groups[0]["lr"]})
            if global_step % CONFIG["save_every"] == 0 or global_step == CONFIG["max_steps"]:
                ema.apply(model)
                val_loss = validation_noise_loss(model, scheduler, val_loader,
                                                 2 if CONFIG["run_mode"] == "smoke" else 20)
                ema.restore(model); history[-1]["val_noise_loss"] = val_loss
                pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
                checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                              "scheduler": lr_scheduler.state_dict(), "scaler": scaler.state_dict(),
                              "ema": ema.state_dict(), "sampler": sampler.state_dict(),
                              "global_step": global_step, "rng_state": rng_state(), "config": run_config}
                atomic_torch_save(checkpoint, run_dir / f"step_{global_step:06d}.pt")
                atomic_torch_save(checkpoint, run_dir / "last.pt")
                save_preview(model, ema, scheduler, run_dir, global_step)
            if global_step >= CONFIG["max_steps"]: break
    progress.close()
    return model, ema, scheduler, run_dir, history


## 6. Đánh giá FID, Inception Score và denoising trace


In [ ]:
def real_fid_state(loader, sample_count, cache_path):
    cache_path = Path(cache_path)
    metric = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    metric.persistent(True)
    if cache_path.exists():
        metric.load_state_dict(torch.load(cache_path, map_location=DEVICE))
        return metric.state_dict()
    seen = 0
    for real in tqdm(loader, desc="cache real FID"):
        real = ((real.to(DEVICE) + 1) / 2).clamp(0, 1)
        take = min(len(real), sample_count - seen)
        metric.update(real[:take], real=True); seen += take
        if seen >= sample_count: break
    atomic_torch_save(metric.state_dict(), cache_path)
    return metric.state_dict()


@torch.no_grad()
def evaluate_generation(model, ema, scheduler, run_dir):
    eval_count = CONFIG["eval_samples"]
    cache = (Path(CONFIG["output_root"]) / "A2" / "cache" /
             f"fid_real_32px_{eval_count}_{MANIFEST_HASH[:12]}.pt")
    state = real_fid_state(test_loader, eval_count, cache)
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
    fid.persistent(True); fid.load_state_dict(state)
    inception = InceptionScore(normalize=True).to(DEVICE)
    ema.apply(model); model.eval()
    generated, generator = 0, torch.Generator().manual_seed(CONFIG["seed"] + 2026)
    start = time.perf_counter()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    while generated < eval_count:
        count = min(CONFIG["batch_size"], eval_count - generated)
        initial = torch.randn(count, 3, 32, 32, generator=generator)
        images, _ = sample_images(model, scheduler, initial, CONFIG["eval_inference_steps"])
        normalized = ((images + 1) / 2).clamp(0, 1)
        fid.update(normalized, real=False); inception.update(normalized)
        generated += count
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    fid_value = fid.compute().item(); is_mean, is_std = inception.compute()
    trace_initial = torch.randn(1, 3, 32, 32, generator=torch.Generator().manual_seed(CONFIG["seed"] + 7))
    _, trace = sample_images(model, scheduler, trace_initial, CONFIG["eval_inference_steps"], True)
    ema.restore(model)
    figure, axes = plt.subplots(1, len(trace), figsize=(2 * len(trace), 2))
    for axis, image in zip(np.asarray(axes).ravel(), trace):
        axis.imshow(((image[0] + 1) / 2).clamp(0, 1).permute(1, 2, 0)); axis.axis("off")
    figure.tight_layout(); figure.savefig(Path(run_dir) / "denoising_trace.png", dpi=160); plt.close(figure)
    return {"run_mode": CONFIG["run_mode"], "fid": fid_value,
            "inception_score_mean": is_mean.item(), "inception_score_std": is_std.item(),
            "generated_samples": eval_count, "inference_steps": CONFIG["eval_inference_steps"],
            "sampling_seconds": elapsed, "seconds_per_image": elapsed / eval_count,
            "peak_gpu_memory_mb": (torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else 0.0),
            **count_parameters(model)}


summary = []
for experiment_id in selected:
    run_dir = Path(CONFIG["output_root"]) / "A2" / CONFIG["run_mode"] / experiment_id
    existing = completed_metrics(run_dir, CONFIG["run_mode"])
    if existing is not None and not CONFIG["evaluate_only"]:
        summary.append({"experiment_id": experiment_id, **existing})
        continue
    model, ema, scheduler, run_dir, history = train_diffusion(experiment_id)
    metrics = evaluate_generation(model, ema, scheduler, run_dir)
    json_dump(metrics, run_dir / "metrics.json")
    summary.append({"experiment_id": experiment_id, **metrics})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

OUTPUT_BASE = Path(CONFIG["output_root"]) / "A2" / CONFIG["run_mode"]
summary_frame = pd.DataFrame(summary).sort_values("experiment_id")
summary_frame.to_csv(OUTPUT_BASE / "summary.csv", index=False)
display(summary_frame)


## 7. Khung phân tích paper

1. **Phương pháp:** giải thích forward noising, epsilon prediction loss, reverse process và vai trò của beta schedule.
2. **Thực nghiệm paper:** dataset, baseline, likelihood/sample-quality metrics và phát hiện liên quan cosine schedule/learned variance.
3. **Hạn chế:** chi phí sampling, độ nhạy compute/model size và giới hạn của FID/IS.
4. **Đối chiếu:** notebook giữ U-Net, seed, data và budget cố định; chỉ thay schedule. Nêu rõ độ phân giải, subset, fixed variance và số mẫu khác paper.

Không so trực tiếp FID với bảng paper. Loss dự đoán noise là diagnostic, không thay thế metric chất lượng sinh ảnh.
